# Example 04: Tool Calling（工具调用 / 函数调用）

单次工具调用 + 多轮工具循环，直到模型有足够信息给出最终答案

**Demonstrates:**
- Defining tools (function schemas)
- Single tool call: model selects and invokes a tool
- Multi-turn tool loop: model calls tools until it has enough info to answer

**Prerequisites:**
```bash
pip install openai
# Start the Hy3 server with:
#   --tool-call-parser hy_v3 --enable-auto-tool-choice   (vLLM)
#   --tool-call-parser hunyuan                            (SGLang)
```

In [ ]:
import json
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ.get("HY3_BASE_URL", "http://127.0.0.1:8000/v1"),
    api_key=os.environ.get("HY3_API_KEY", "EMPTY"),
)

MODEL = os.environ.get("HY3_MODEL", "hy3")

## Tool Definitions（工具定义）

使用 JSON Schema 描述工具的参数，模型根据用户意图决定调用哪个工具。

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. 'Beijing'"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_time",
            "description": "Get the current local time for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                },
                "required": ["city"],
            },
        },
    },
]

# --- Simulated tool implementations (replace with real API calls) ---

def get_weather(city: str, unit: str = "celsius") -> dict:
    data = {
        "Beijing":  {"celsius": 28, "condition": "Sunny"},
        "New York": {"celsius": 22, "condition": "Partly cloudy"},
        "London":   {"celsius": 15, "condition": "Overcast"},
    }
    info = data.get(city, {"celsius": 20, "condition": "Unknown"})
    temp = info["celsius"] if unit == "celsius" else round(info["celsius"] * 9 / 5 + 32)
    return {"city": city, "temperature": temp, "unit": unit, "condition": info["condition"]}

def get_time(city: str) -> dict:
    times = {"Beijing": "14:35", "New York": "02:35", "London": "07:35"}
    return {"city": city, "local_time": times.get(city, "12:00")}

def call_tool(name: str, arguments: str) -> str:
    """Dispatch a tool call by name. Raises ValueError for unregistered tools."""
    args = json.loads(arguments)
    if name == "get_weather":
        return json.dumps(get_weather(**args))
    elif name == "get_time":
        return json.dumps(get_time(**args))
    # Raise so the caller surfaces a clear error rather than feeding a silent
    # {"error": ...} result back to the model, which may cause hallucination.
    raise ValueError(f"Unknown tool requested by model: {name!r}")

## Single Tool Call（单次工具调用）

第一次请求：模型返回 `finish_reason="tool_calls"` + 调用参数。  
执行工具后，将结果放入 messages，第二次请求得到最终答案。

In [ ]:
messages = [{"role": "user", "content": "What's the weather like in Beijing right now?"}]

# Round 1: model decides which tool to call
response = client.chat.completions.create(
    model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto",
    temperature=0.9, top_p=1.0,
)

choice = response.choices[0]
print(f"Finish reason: {choice.finish_reason}")

if choice.finish_reason == "tool_calls":
    messages.append(choice.message)
    for tc in choice.message.tool_calls:
        print(f"→ Model called: {tc.function.name}({tc.function.arguments})")
        result = call_tool(tc.function.name, tc.function.arguments)
        print(f"← Tool returned: {result}")
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    # Round 2: generate final answer using tool results
    final = client.chat.completions.create(
        model=MODEL, messages=messages, tools=TOOLS, temperature=0.9, top_p=1.0,
    )
    print(f"\nFinal answer: {final.choices[0].message.content}")

**Sample output:**
```
Finish reason: tool_calls
→ Model called: get_weather({"city": "Beijing", "unit": "celsius"})
← Tool returned: {"city": "Beijing", "temperature": 28, "unit": "celsius", "condition": "Sunny"}

Final answer: The current weather in Beijing is sunny with a temperature of 28°C.
```

## Multi-turn Tool Loop（多轮工具循环）

模型可能在一轮中调用多个工具（并行），或跨多轮调用工具（串行），
直到 `finish_reason="stop"` 才输出最终答案。

In [ ]:
messages = [
    {
        "role": "user",
        "content": "I'm planning a call between teams in Beijing and New York. "
                   "What's the weather and local time in both cities?",
    }
]

for round_num in range(5):  # guard against infinite loops
    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto",
        temperature=0.9, top_p=1.0,
    )
    choice = response.choices[0]
    messages.append(choice.message)

    if choice.finish_reason == "stop":
        print(f"Final answer (after {round_num} tool round(s)):")
        print(choice.message.content)
        break

    if choice.finish_reason == "tool_calls":
        print(f"Round {round_num + 1}: {len(choice.message.tool_calls)} tool call(s)")
        for tc in choice.message.tool_calls:
            result = call_tool(tc.function.name, tc.function.arguments)
            print(f"  → {tc.function.name}({tc.function.arguments})")
            print(f"  ← {result}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

**Sample output:**
```
Round 1: 4 tool call(s)
  → get_weather({"city": "Beijing"})
  ← {"city": "Beijing", "temperature": 28, "unit": "celsius", "condition": "Sunny"}
  → get_time({"city": "Beijing"})
  ← {"city": "Beijing", "local_time": "14:35"}
  → get_weather({"city": "New York"})
  ← {"city": "New York", "temperature": 22, "unit": "celsius", "condition": "Partly cloudy"}
  → get_time({"city": "New York"})
  ← {"city": "New York", "local_time": "02:35"}

Final answer (after 1 tool round(s)):
Here's the current status for both cities:

**Beijing**: 28°C, Sunny — local time 14:35
**New York**: 22°C, Partly cloudy — local time 02:35

Note: with a 12-hour time difference, New York is in the middle of the night (02:35).
A good overlap for both teams might be early morning New York time (e.g. 09:00 EST = 22:00 BJT).
```